<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/supervision_5_funciones_compartible.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cinco funciones de Supervision

Notebook ejecutable para demostrar cinco funciones por separado: anotadores, seguimiento, línea de conteo, zonas y mapa de calor. Ejecuta las celdas en orden. En Colab puedes subir un video; en Jupyter local coloca video_entrada.mp4 junto al notebook.

In [1]:
%pip install -q -U ultralytics supervision trackers

In [2]:
import os, cv2, numpy as np
import supervision as sv
from IPython.display import display
from PIL import Image
from ultralytics import YOLO
from trackers import ByteTrackTracker

VIDEO_IN = "people_walking.mp4"
INICIO_S, DURACION_S, CONF = 0, 10, 0.30
CLASES = [0, 1, 2, 3, 5, 7]  # persona, bici, auto, moto, bus, camión

if not os.path.exists(VIDEO_IN):
    try:
        from google.colab import files
        cargados = files.upload()
        if cargados: VIDEO_IN = next(iter(cargados))
    except ImportError:
        pass
if not os.path.exists(VIDEO_IN):
    import urllib.request
    url = "https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4"
    urllib.request.urlretrieve(url, VIDEO_IN)

modelo = YOLO("yolo11n.pt")
info = sv.VideoInfo.from_video_path(VIDEO_IN)
print(f"Video: {VIDEO_IN} | {info.width}×{info.height} | {info.fps:.1f} fps")

def cuadros_con_detecciones():
    inicio = int(INICIO_S * info.fps)
    fin = inicio + int(DURACION_S * info.fps)
    for frame in sv.get_video_frames_generator(VIDEO_IN, start=inicio, end=fin):
        resultado = modelo(frame, conf=CONF, verbose=False)[0]
        det = sv.Detections.from_ultralytics(resultado)
        if det.class_id is not None: det = det[np.isin(det.class_id, CLASES)]
        yield frame, det

def mostrar(frame, titulo):
    print(titulo)
    display(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


Saving people-walking.mp4 to people-walking (1).mp4
Video: people-walking (1).mp4 | 1920×1080 | 25.0 fps


## 1. Anotadores

Dibuja cajas y etiquetas con clase y confianza.

In [6]:
cajas = sv.BoxAnnotator(thickness=2)
etiquetas = sv.LabelAnnotator(text_scale=0.6, text_thickness=1)

# Definir el archivo de salida para guardar todo el video procesado
VIDEO_OUT = "resultado_anotadores.mp4"

print("Procesando todo el video... Por favor espera.")
with sv.VideoSink(target_path=VIDEO_OUT, video_info=info) as sink:
    for cuadro, det in cuadros_con_detecciones():
        labels = [f"{modelo.names[int(c)]} {p:.2f}" for c, p in zip(det.class_id, det.confidence)]
        salida = cajas.annotate(scene=cuadro.copy(), detections=det)
        salida = etiquetas.annotate(scene=salida, detections=det, labels=labels)
        sink.write_frame(frame=salida)

print(f"¡Proceso completado! El video resultante se ha guardado en: {VIDEO_OUT}")

Procesando todo el video... Por favor espera.
¡Proceso completado! El video resultante se ha guardado en: resultado_anotadores.mp4


## 2. Seguimiento

Asigna un ID persistente a cada objeto y dibuja su trayectoria.

In [7]:
tracker = ByteTrackTracker()
trazas, cajas = sv.TraceAnnotator(trace_length=40, thickness=2), sv.BoxAnnotator(thickness=2)
etiquetas = sv.LabelAnnotator(text_scale=0.6, text_thickness=1)

VIDEO_OUT_TRACK = "resultado_seguimiento.mp4"

print("Procesando el seguimiento en todo el video... Por favor espera.")
with sv.VideoSink(target_path=VIDEO_OUT_TRACK, video_info=info) as sink:
    for cuadro, det in cuadros_con_detecciones():
        seguidas = tracker.update(det)
        seguidas = seguidas[seguidas.tracker_id != -1]
        salida = trazas.annotate(scene=cuadro.copy(), detections=seguidas)
        salida = cajas.annotate(scene=salida, detections=seguidas)
        salida = etiquetas.annotate(scene=salida, detections=seguidas, labels=[f"#{i}" for i in seguidas.tracker_id])
        sink.write_frame(frame=salida)

print(f"¡Proceso de seguimiento completado! Guardado en: {VIDEO_OUT_TRACK}")

Procesando el seguimiento en todo el video... Por favor espera.
¡Proceso de seguimiento completado! Guardado en: resultado_seguimiento.mp4


## 3. Línea de conteo

Cuenta cruces por una línea horizontal a media altura.

In [8]:
linea = sv.LineZone(start=sv.Point(0, info.height//2), end=sv.Point(info.width-1, info.height//2))
linea_ann, tracker, cajas = sv.LineZoneAnnotator(thickness=2), ByteTrackTracker(), sv.BoxAnnotator(thickness=2)

VIDEO_OUT_LINE = "resultado_linea_conteo.mp4"

print("Procesando el conteo de línea en todo el video... Por favor espera.")
with sv.VideoSink(target_path=VIDEO_OUT_LINE, video_info=info) as sink:
    for cuadro, det in cuadros_con_detecciones():
        seguidas = tracker.update(det)
        seguidas = seguidas[seguidas.tracker_id != -1]
        linea.trigger(seguidas)
        salida = cajas.annotate(scene=cuadro.copy(), detections=seguidas)
        salida = linea_ann.annotate(frame=salida, line_counter=linea)
        sink.write_frame(frame=salida)

print(f"¡Conteo de línea completado! Guardado en: {VIDEO_OUT_LINE} | Entran: {linea.in_count}, Salen: {linea.out_count}")

Procesando el conteo de línea en todo el video... Por favor espera.
¡Conteo de línea completado! Guardado en: resultado_linea_conteo.mp4 | Entran: 6, Salen: 2


## 4. Zonas

Dibuja un polígono y resalta los objetos que están dentro.

In [9]:
poligono = np.array([[int(x*(info.width-1)), int(y*(info.height-1))] for x,y in [(0.15,0.25),(0.85,0.25),(0.85,0.95),(0.15,0.95)]])
zona = sv.PolygonZone(polygon=poligono)
zona_ann, cajas = sv.PolygonZoneAnnotator(zone=zona, color=sv.Color.RED, thickness=2), sv.BoxAnnotator(thickness=2)

VIDEO_OUT_ZONE = "resultado_zonas.mp4"

print("Procesando las zonas en todo el video... Por favor espera.")
with sv.VideoSink(target_path=VIDEO_OUT_ZONE, video_info=info) as sink:
    for cuadro, det in cuadros_con_detecciones():
        dentro = zona.trigger(det)
        salida = cajas.annotate(scene=cuadro.copy(), detections=det[dentro])
        salida = zona_ann.annotate(scene=salida)
        sink.write_frame(frame=salida)

print(f"¡Proceso de zonas completado! Guardado en: {VIDEO_OUT_ZONE}")

Procesando las zonas en todo el video... Por favor espera.
¡Proceso de zonas completado! Guardado en: resultado_zonas.mp4


## 5. Mapa de calor

Acumula las posiciones detectadas para mostrar por dónde pasan los objetos.

In [10]:
calor, tracker = sv.HeatMapAnnotator(opacity=0.6, radius=30), ByteTrackTracker()

VIDEO_OUT_HEATMAP = "resultado_mapa_calor.mp4"

print("Procesando el mapa de calor en todo el video... Por favor espera.")
with sv.VideoSink(target_path=VIDEO_OUT_HEATMAP, video_info=info) as sink:
    for cuadro, det in cuadros_con_detecciones():
        seguidas = tracker.update(det)
        seguidas = seguidas[seguidas.tracker_id != -1]
        salida = calor.annotate(scene=cuadro.copy(), detections=seguidas)
        sink.write_frame(frame=salida)

print(f"¡Mapa de calor completado! Guardado en: {VIDEO_OUT_HEATMAP}")

Procesando el mapa de calor en todo el video... Por favor espera.
¡Mapa de calor completado! Guardado en: resultado_mapa_calor.mp4
